### Вариант 2 (рабочий): из HTML читаем

In [20]:
from bs4 import BeautifulSoup
import json
import re

html_file = "/Users/konstantin/Documents/Заказ оформлен.html"
with open(html_file, "r", encoding="utf-8") as file:
    soup = BeautifulSoup(file, "html.parser")
    
raw_text = ''
for script in soup.find_all("script"):
    if len(script.text) > 100 and script.text.startswith("window.__REACT_QUERY_STATE__"):
        raw_text = script.text[29:]

# чиcтим чтобы влезло в JSON
match = re.search(r'({.*})', raw_text, re.DOTALL)
if not match:
    raise ValueError("No JSON-like block found in the file.")
json_like_text = match.group(1)
cleaned_json_text = (
    json_like_text
    .replace("undefined", "null")  # JS -> JSON
    .replace("True", "true")
    .replace("False", "false")
)

try:
    data = json.loads(cleaned_json_text)
except json.JSONDecodeError as e:
    raise ValueError(f"JSON parsing error: {e}")

# json[json.find("Рис")-50:json.find("Рис")+100]
titles = [x['name'] for x in data['queries'][1]['state']['data']['calculation']['items']]

titles = [x.replace('\u200b', '') for x in titles]

for i, title in enumerate(titles, 1):
    print(f"{i}. {title}")


1. Энергетический напиток Red Bull
2. Пирожное протеиновое ProteinRex Классическое брауни
3. Груши Китайские
4. Грудка куриная запечённая 2 шт. «Из Лавки»
5. Сэндвич с тунцом и маринованным луком «Из Лавки»
6. Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
7. Печенье протеиновое Fit Kit фисташковая кунафа
8. Сырок творожный глазированный «Из Лавки» обезжиренный 8%
9. Чипсы зерновые Флайчипсы «Из Лавки» 3 сыра и томат
10. Салат из кальмаров командорских и моркови по-корейски Creative Kitchen
11. Мясо курицы сушёное «Из Лавки» с горчицей
12. Молоко 2,5% «Домик в деревне» ультрапастеризованное
13. Яблоки Гренни


## Справочник калорийности

In [21]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# Define scope
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

# Load credentials
creds = ServiceAccountCredentials.from_json_keyfile_name('credentials.json', scope)
client = gspread.authorize(creds)

# Open Google Sheet by name or URL
spreadsheet = client.open("Калории")

# Select worksheet/tab by name
worksheet = spreadsheet.worksheet("Reference")

# Get all values as list of rows
data = worksheet.get_all_values()

reference = pd.DataFrame(data[1:], columns=data[0])

reference


from datetime import datetime
import numpy as np

# Match products with reference records
update = reference[reference['продукт'].isin(titles)]

not_found = set(list(titles)).difference(set(reference['продукт']))
print("Not found: ")
for i in not_found:
    print(i)

# Insert date columns
current_date = datetime.today().strftime('%Y%m%d')
# current_date = '20250923'; print("SETTING manual date")
update.insert(0, 'дата', current_date)
update.insert(len(update.columns), 'активность', np.nan)

# Use Formulas instead of constants
update.loc[:,'ккал'] = 0.0
update.loc[:,'б на порцию'] = 0.0
update.loc[:,'ж на порцию'] = 0.0
update.loc[:,'у на порцию'] = 0.0

# Update types
update.loc[:,'вес'] = update['вес'].astype(float)
update.loc[:,'ккал на 100г'] = update['ккал на 100г'].astype(float)
update.loc[:,'б на 100г'] = update['б на 100г'].astype(float)
update.loc[:,'ж на 100г'] = update['ж на 100г'].astype(float)
update.loc[:,'у на 100г'] = update['у на 100г'].astype(float)

# Compute totals
totals = update.iloc[:,2:].sum()
totals_row = pd.DataFrame([[current_date, 'Total'] + totals.tolist()], columns=update.columns)
update = pd.concat([update, totals_row], ignore_index=True)

# Append the blank row
blank_row = {col:np.nan for col in update.columns}
blank_row['дата']=current_date
update = pd.concat([update, pd.DataFrame([blank_row])], ignore_index=True)

update

Not found: 


,дата,продукт,вес,ккал на 100г,ккал,б на 100г,ж на 100г,у на 100г,б на порцию,ж на порцию,у на порцию,активность
0,20251017,Грудка куриная запечённая 2 шт. «Из Лавки»,1.4,167.0,0.0,31.0,4.7,0.0,0.0,0.0,0.0,NaN
1,20251017,Груши Китайские,5.0,47.0,0.0,0.4,0.3,10.4,0.0,0.0,0.0,NaN
2,20251017,"Молоко 2,5% «Домик в деревне» ультрапастеризов...",9.5,53.0,0.0,2.9,2.5,4.7,0.0,0.0,0.0,NaN
3,20251017,Мороженое протеиновое Bombbar фисташковое без ...,0.8,124.0,0.0,6.3,4.5,14.4,0.0,0.0,0.0,NaN
4,20251017,Мясо курицы сушёное «Из Лавки» с горчицей,0.35,270.0,0.0,57.0,4.0,2.0,0.0,0.0,0.0,NaN
5,20251017,Печенье протеиновое Fit Kit фисташковая кунафа,0.4,275.0,0.0,27.5,12.5,17.5,0.0,0.0,0.0,NaN
6,20251017,Пирожное протеиновое ProteinRex Классическое б...,0.5,385.0,0.0,12.0,28.0,18.0,0.0,0.0,0.0,NaN
7,20251017,Салат из кальмаров командорских и моркови по-к...,1.1,175.0,0.0,6.0,13.0,9.0,0.0,0.0,0.0,NaN
8,20251017,Сырок творожный глазированный «Из Лавки» обезж...,0.45,254.0,0.0,11.3,8.0,34.2,0.0,0.0,0.0,NaN
9,20251017,Сэндвич с тунцом и маринованным луком «Из Лавки»,1.9,155.0,0.0,10.0,3.0,22.0,0.0,0.0,0.0,NaN


In [ ]:
from gspread_dataframe import get_as_dataframe, set_with_dataframe

upd_worksheet = spreadsheet.worksheet("2025 New")

existing = get_as_dataframe(upd_worksheet, evaluate_formulas=True, header=0)
first_row = len(existing) + 5

# в первой строке пишем названия колонок, поэтому в формуле надо сдвинуть номер строки
header_shift = 1

# в последних строках total и пустая строка - туда формуклу не пишем
footer_shift = 2

def create_formulas(weight_column, calories_100_column, calories_total_column):
    calories_formula = ['={}{} * {}{}'.format(weight_column, str(x + first_row + header_shift), calories_100_column, str(x + first_row + header_shift)) for x in list(update.index)]
    calories_formula = calories_formula[:len(calories_formula)-footer_shift]
    calories_formula = calories_formula + [''] * footer_shift
    return calories_formula

update['ккал'] = create_formulas(weight_column = 'C', calories_100_column = 'D', calories_total_column = 'ккал')
update['б на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'F', calories_total_column = 'б на порцию')
update['ж на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'G', calories_total_column = 'ж на порцию')
update['у на порцию'] = create_formulas(weight_column = 'C', calories_100_column = 'H', calories_total_column = 'у на порцию')

update.columns = [current_date if x=='дата' else x for x in list(update.columns)]

# собираем формулы в Total
total_row_ref = update['продукт']=='Total'
total_row_num = list(update[total_row_ref].index)[0]

update.loc[total_row_num, 'ккал'] = '=SUM(E{}:E{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'б на порцию'] = '=SUM(I{}:I{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'ж на порцию'] = '=SUM(J{}:J{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'у на порцию'] = '=SUM(K{}:K{})'.format(first_row + header_shift, first_row + header_shift + total_row_num - 1)
update.loc[total_row_num, 'ккал на 100г'] = ''
update.loc[total_row_num, 'активность'] = ''

# Step 5: Append new data
set_with_dataframe(
    upd_worksheet, 
    update, 
    row=first_row,
    col=1,
    include_column_header=True)

# Test before use
# worksheet.sort((1, 'desc'))

In [117]:
import pytesseract
import os
import re
from PIL import Image

def get_most_recent_file(directory):
    
    # List all files    
    files = [os.path.join(directory, f) for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith('.png')]

    # Get the most recent
    most_recent_file = max(files, key=os.path.getctime)
    
    return most_recent_file


# Get the screenshot
directory_path = "/Users/konstantin/Documents"
recent_file = get_most_recent_file(directory_path)
print("Found the screenshot file", recent_file)

# Load image
image = Image.open(recent_file)

# Set language to Russian ('rus')
text = pytesseract.image_to_string(image, lang='rus+eng')

UNINFORMATIVE_RE = re.compile(
    r'[\d.,]+\s?(₽|р|Р|P|руб\.?)|Срок годности',
    re.IGNORECASE
)

def extract_product_title(lines):
    pattern = r'^\d+$'
    return sorted([line.strip() for line in lines if not UNINFORMATIVE_RE.search(line) and len(line)>5 and not re.fullmatch(pattern, line.replace(' ','').strip())])

# Example usage
titles = extract_product_title(text.split('\n'))

for i,x in enumerate(titles):
    print(i, ': ', repr(x))


Found the screenshot file /Users/konstantin/Documents/Screenshot 2025-07-20 at 03.39.56.png
0 :  'class GraphState(TypedDict) :'
1 :  'from langchain_core.messages import AnyMessage'
2 :  'from langgraph.graph.message import add_messages'
3 :  'from typing import Annotated'
4 :  'from typing_extensions import TypedDict'
5 :  'messages: Annotated[list[AnyMessage], add_messages ]'


In [2]:
# Manual merging
for pair in [(2, 6)]:
    joined = ' '.join([titles[pair[0]], titles[pair[1]]])
    print("Merged: ",joined)
    titles[pair[0]] = joined
    titles[pair[1]] = ''

titles = [x for x in titles if x != '']

for i,x in enumerate(titles):
    print(i, ': ', x)

Merged:  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
0 :  Батончик протеиновый ProteinRex кокос
1 :  Дыня нарезанная кубиками «Из Лавки»
2 :  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
3 :  Продукт творожный «Даниссимо» с сочным киви 5,5%
4 :  Салат оливье с курицей «Из Лавки»
5 :  Сэндвич стунцом и маринованным луком «Из Лавки»
